# 05_07 Train the GPT

*📖 Book: section 5.2 (training); the learning-rate schedule is in appendix D*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → [model learns] → model writes → MCP tool
> ```
>
> You are here: **model learns**
>
> Project files this connects to:
>
> - `src/homer_ai/train_homer.py`
> - `src/homer_ai/gpt.py`
>
> **This notebook:** The course has two models, and the GPT of chapter 4 has its own training loop, `train_gpt`. What's different in it, why dropout 0.3, and how do you train your own?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How `train_gpt` feeds the model: batches of random windows
- Why the scores are flattened before the loss
- How the cosine schedule ticks once per step, not once per epoch
- What weight decay and dropout do, and why the GPT needs dropout 0.3
- How to train your own GPT with one command, and read what it prints

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **batch** | several pieces of text the model works on in one step, stacked into one tensor | 16 windows of 64 letters |
| **weight decay** | at every step, the optimizer also pulls each weight a tiny bit towards 0, so a weight only stays big if the loss keeps asking for it | `weight_decay=0.01` |
| **held-out loss** | another name for the validation loss: the loss on text held out of training | `2.27` on the last 10% |

Words you've met before: **window** ([02_03](../02_text_to_numbers/02_03_bpe.ipynb)).

## 3. The idea

### ❓ We already have a training loop. Why does the GPT get its own?

Because there are two models in this course, and each has its own loop in `src/homer_ai/train_homer.py`. Everything in 05_01 to 05_04 used the small one. This notebook is about the other one.

| | The small model | The GPT |
|---|---|---|
| Code | `model.py`, 11,840 weights | `gpt.py`, 835,072 weights ([04_05](../04_building_the_model/04_05_the_full_model.ipynb)) |
| Loop | `train()` | `train_gpt()` |
| One step reads | one chunk of 24 letters | a batch: 16 windows of 64 letters |
| Which pieces | back to back, in order | picked at random |
| Trains on | all of Book 1 | the first 90% ([05_05](05_05_validation.ipynb)) |
| Schedule ticks | once per epoch | once per step |
| Dropout | none | 0.3 |
| Checks on hidden text | no | every 20 epochs |
| Command | `uv run homer-train` | `uv run homer-train --model gpt` |
| Writes | `src/homer_ai/homer_model.pt` | `src/homer_ai/homer_gpt.pt` |

The four steps inside are the ones you know: guess, measure the loss, work out the nudges, nudge. Here's the whole inner loop of `train_gpt`:

```python
inputs, targets = random_batch(train_data, size, batch_size)       # step 1
logits = model(inputs)
loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())  # step 2
optimizer.zero_grad()
loss.backward()
optimizer.step()
scheduler.step()                                                   # step 3
```

Three lines are new, and steps 1 to 3 below take them one at a time. Then weight decay and dropout, which sit in the lines around the loop.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: what's a batch of random windows?

The small model's loop walks along the text, one chunk after the other. `random_batch` does it differently. It picks `batch_size` starting places at random, cuts a window at each one, and stacks the windows into one tensor. The targets are the same windows moved one letter on, as in [02_04](../02_text_to_numbers/02_04_sliding_window.ipynb).

Let's shrink it so we can read it. The text is the first 23 characters of Book 1 (`␣` is a space), the windows are 4 letters long, and the batch is 3 windows:

```
ἄ ν δ ρ α ␣ μ ο ι ␣ ἔ ν ν ε π ε , ␣ μ ο ῦ σ α
            └─────┘                          a window starting at μ
inputs :    μ  ο  ι  ␣
targets:    ο  ι  ␣  ἔ                       read it column by column
```

Why random? With windows cut back to back, the model would see the same 277 windows of the training text in every epoch. A window may start at any letter, so there are 17,665 different ones, and every step shows the model a fresh mix.

👉 **Predict first:** what shape will `inputs` have: 3 windows of 4 letters?

In [ ]:
import torch

from homer_ai.train_homer import BOOKS_DIR, load_data, random_batch

data, tokenizer, text = load_data(BOOKS_DIR)  # Book 1 as numbers
line = data[:23]
print(repr(tokenizer.decode(line.tolist())))

torch.manual_seed(0)
inputs, targets = random_batch(line, context_length=4, batch_size=3)
print("inputs:", tuple(inputs.shape), " targets:", tuple(targets.shape))
print()
for window, answers in zip(inputs, targets, strict=True):
    print("inputs :", "  ".join(tokenizer.decode(window.tolist()).replace(" ", "␣")))
    print("targets:", "  ".join(tokenizer.decode(answers.tolist()).replace(" ", "␣")))
    print()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Both are `(3, 4)`: 3 windows, 4 letters each. That first number is the batch.

```
inputs : ι  ␣  ἔ  ν
targets: ␣  ἔ  ν  ν
```

Each targets row is its inputs row moved one letter on. And the windows overlap (the second one starts one letter after the first), which back-to-back chunks never do. The real run uses `context_length=64` and `batch_size=16`, so one step reads 16 × 64 = 1,024 letters.

</details>

### ❓ Step 2: why `flatten(0, 1)` before the loss?

The GPT answers every letter of every window at once. For our batch that's 3 × 4 = 12 questions ("what comes after this letter?"), and for each one it gives 137 scores. So the logits have **three** dimensions: `(3, 4, 137)`.

`F.cross_entropy` wants something simpler: one row of scores per question, and one right answer per question. It doesn't care which window a question came from. So we lay the 12 questions out in one long list:

```
logits   (3, 4, 137)  → .flatten(0, 1) →  (12, 137)   12 questions, 137 scores each
targets  (3, 4)       → .flatten()     →  (12,)       12 right answers
```

`flatten(0, 1)` merges dimension 0 and dimension 1 (windows and letters) into one, and leaves the 137 scores alone. The small model never needed this: it reads one chunk with no batch dimension, so its logits are already `(24, 137)`.

We try it on a small untrained GPT (2 blocks, 64 numbers per letter), so the cell runs at once.

👉 **Predict first:** the model is untrained. About what loss do you expect? (Random guessing among 137 gives `ln(137) = 4.92`.)

In [ ]:
import math

import torch.nn.functional as F

from homer_ai.gpt import GPTConfig, GPTModel

SMALL = {"context_length": 32, "emb_dim": 64, "n_heads": 4, "n_layers": 2}

torch.manual_seed(42)
small_gpt = GPTModel(GPTConfig(vocab_size=tokenizer.vocab_size, drop_rate=0.0, **SMALL))

logits = small_gpt(inputs)
print("logits: ", tuple(logits.shape))
print("flat logits:", tuple(logits.flatten(0, 1).shape))
print("flat targets:", tuple(targets.flatten().shape))
loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
print("loss:", round(loss.item(), 3), " ln(137):", round(math.log(137), 3))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`(3, 4, 137)` becomes `(12, 137)`, and the targets become 12 numbers. The loss is one number, the average over all 12 questions: about 5.16, close to the 4.92 of pure guessing. (A new model's scores aren't perfectly even, so it's a little worse than guessing. Training fixes that within a few steps.)

</details>

### ❓ Step 3: when does the learning rate change?

In [05_02](05_02_learning_rate.ipynb) the cosine schedule moved one notch per **epoch**. `train_gpt` moves it one notch per **step**, so the learning rate slides down smoothly and doesn't go down in 200 stairs. For that, the schedule has to know how many steps there will be in all:

```python
steps_per_epoch = len(train_data) // (size * batch_size)
scheduler = CosineAnnealingLR(optimizer, T_max=epochs * steps_per_epoch)
```

With random windows there's no real "pass through the text" any more. So here an **epoch** just means: as many steps as it takes to read about as many letters as the training text has.

The settings are in `GPT_SETTINGS` and `GPT_TRAINING`, at the top of `train_homer.py`. We read them from there and do the sums. The schedule below is a real `CosineAnnealingLR`. It has only one dummy number to look after, so nothing is trained.

👉 **Predict first:** 17,730 training letters, and 1,024 letters per step. How many steps is one epoch? And 200 epochs?

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

from homer_ai.train_homer import GPT_SETTINGS, GPT_TRAINING, split_ids

train_data, val_data = split_ids(data)  # the first 90%, the last 10% (05_05)
size, batch_size = GPT_SETTINGS["context_length"], GPT_TRAINING["batch_size"]
steps_per_epoch = len(train_data) // (size * batch_size)
total_steps = GPT_TRAINING["epochs"] * steps_per_epoch
print("training letters:", len(train_data), " letters per step:", size * batch_size)
print("steps per epoch: ", steps_per_epoch, " steps in all:", total_steps)

dummy = torch.optim.AdamW([torch.zeros(1, requires_grad=True)], lr=GPT_TRAINING["lr"])
schedule = CosineAnnealingLR(dummy, T_max=total_steps)
for step in range(total_steps + 1):
    if step % 850 == 0:
        epoch = step // steps_per_epoch
        print(f"step {step:4} (epoch {epoch:3}): lr {schedule.get_last_lr()[0]:.5f}")
    dummy.step()
    schedule.step()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

17 steps per epoch (17 × 1,024 = 17,408 letters, about one pass), so 200 epochs is 3,400 steps. The learning rate starts at its peak, 0.001, is exactly half (0.0005) at step 1,700, and reaches 0 at step 3,400. It's the curve of [05_02](05_02_learning_rate.ipynb), with 3,400 small notches.

</details>

### ❓ Step 4: what's weight decay?

The optimizer line of `train_gpt` has one setting we haven't met:

```python
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
```

**Weight decay** is a small pull towards 0. At every step, on top of the usual nudge, AdamW shrinks each weight by a tiny share of itself: `learning rate × weight_decay × weight`. A weight the loss really needs gets pushed back up by its gradient at the next step. A weight that grew big only to memorise one odd spot of the text slowly fades. So it's a mild brake on memorising.

To see the pull by itself, we take two weights whose gradient is exactly 0, so the loss asks for no nudge at all. (The learning rate here is 0.1, so the numbers are easy to check.)

👉 **Predict first:** a weight of 1.0, learning rate 0.1, weight decay 0.01. What is it after one step?

In [ ]:
weights = torch.tensor([1.0, -2.0], requires_grad=True)
optimizer = torch.optim.AdamW([weights], lr=0.1, weight_decay=0.01)

loss = (weights * 0).sum()  # a loss that doesn't care about the weights: gradient 0
loss.backward()
optimizer.step()
print("gradient:", weights.grad.tolist())
print("after one step:", [round(w, 4) for w in weights.tolist()])
default = torch.optim.AdamW([weights]).defaults["weight_decay"]
print("AdamW's default weight decay:", default)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The gradient is 0, and still both weights moved towards 0: 1.0 became 0.999 (it lost `0.1 × 0.01 × 1.0 = 0.001`), and −2.0 became −1.998. Each weight shrinks by the same share, so bigger weights lose more.

The last line tells you something about the small model too. AdamW's default is already 0.01, so `train()` has used the same weight decay all along. `train_gpt` just writes it out.

</details>

### ❓ Step 5: what does dropout 0.3 do while training?

**Dropout** ([03_03](../03_attention/03_03_hiding_the_future.ipynb)) switches off numbers at random while the model trains. With `drop_rate=0.3`, each number has a 3 in 10 chance of becoming 0 at every step, and the ones that stay are scaled up (divided by 0.7), so the total stays about the same. The model can't count on any single number being there, so it has to spread what it knows. That makes learning a text by heart harder.

Dropout is only for training. `model.train()` switches it on, and `model.eval()` switches it off, so measuring and writing use every number.

👉 **Predict first:** ten 1s go through `nn.Dropout(0.3)` in training mode. What comes out? And in eval mode?

In [ ]:
from torch import nn

torch.manual_seed(0)
dropout = nn.Dropout(0.3)
ones = torch.ones(10)

dropout.train()
print("training:", [round(v, 2) for v in dropout(ones).tolist()])
dropout.eval()
print("eval:    ", [round(v, 2) for v in dropout(ones).tolist()])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

In training mode some of the 1s became 0 and the others became 1.43 (that's 1 ÷ 0.7). In this draw 4 of the 10 were switched off. On average it's 3. In eval mode all ten come out as 1, untouched.

</details>

### ❓ Step 6: does dropout really matter? And what's the held-out check?

Let's run `train_gpt` itself and watch. Not the real training (that's 3 minutes, and it's a terminal command), but a small one that takes seconds:

```
text:    the first 4,000 letters of Book 1 (3,600 to train on, 400 held out)
model:   a small GPT: 2 blocks, 64 numbers per letter, reads 32 letters
run 1:   dropout 0.0
run 2:   dropout 0.3     <- the ONE change (same seed, same everything else)
```

Every 20 epochs `train_gpt` prints a line with two losses. The **train loss** is the average over the steps of that epoch, measured while learning. The **held-out loss** comes from `evaluate_gpt`, which is the `evaluate` of [05_05](05_05_validation.ipynb) for the GPT: dropout off (`model.eval()`), no learning (`torch.no_grad()`), on the text the model never trains on. At the end the function returns both losses measured that same careful way.

This small run uses a learning rate of 0.003, so things happen fast. Both runs together take about 20 seconds, and nothing is saved.

👉 **Predict first:** without dropout the train loss will go lower. What will its held-out loss do?

In [ ]:
from homer_ai.train_homer import train_gpt

slice_train, slice_val = split_ids(data[:4000])
print("training letters:", len(slice_train), " held out:", len(slice_val))

final = {}
for drop_rate in (0.0, 0.3):
    print(f"\ndropout {drop_rate}")
    torch.manual_seed(42)
    config = GPTConfig(vocab_size=tokenizer.vocab_size, drop_rate=drop_rate, **SMALL)
    final[drop_rate] = train_gpt(
        GPTModel(config), slice_train, slice_val,
        seed=42, epochs=120, lr=0.003, batch_size=16, eval_every=20,
    )

print()
for drop_rate, losses in final.items():
    print(
        f"dropout {drop_rate}: train loss {losses['final_train_loss']:.2f}"
        f" | held-out loss {losses['val_loss']:.2f}"
    )

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

| | Train loss at the end | Held-out loss, epoch 20 → 120 |
|---|---|---|
| dropout 0.0 | 0.77 | 2.91 → 2.93 → 3.19 → 3.47 → 3.63 → **3.65** |
| dropout 0.3 | 2.29 | 2.96 → 2.93 → 2.93 → 2.91 → 2.91 → **2.90** |

Without dropout the train loss falls to 0.77. If that were all we printed, it would look like a great run. But the held-out loss turns round after epoch 20 and climbs to 3.65. The model is learning its 3,600 letters by heart, and getting worse at every other piece of Greek. That's the overfitting of [05_03](05_03_memorising_vs_understanding.ipynb), caught while it happens.

With dropout 0.3 the train loss stays much higher (2.29), and the held-out loss keeps going down, slowly. It's the better model of the two on text it hasn't seen, 2.90 against 3.65.

That's why `GPT_SETTINGS` says `drop_rate: 0.3`. The real model is far bigger than this one, and Book 1 is still a small text. (Your last digits may differ a little on another machine. The picture won't.)

</details>

## 5. Full example: train your own GPT

### ❓ How do I train the real one?

In a terminal, in the project folder:

```
uv run homer-train --model gpt
```

It runs `main_gpt()` in `train_homer.py`: load Book 1, split it 90/10, build a `GPTModel` from `GPT_SETTINGS`, call `train_gpt` with `GPT_TRAINING`, and save a checkpoint ([05_06](05_06_saving_and_loading.ipynb)). It takes about 3 minutes on a laptop, and prints this (the `…` are losses we left out; the `lr` column is the schedule of step 3):

```
Training a GPT (835,072 weights) on 17730 Greek characters, checking on the last 1971 (Vocab size: 137)...
Epoch 20/200 | train loss: … | held-out loss: … | lr: 0.00098
Epoch 40/200 | train loss: … | held-out loss: … | lr: 0.00090
…
Epoch 200/200 | train loss: … | held-out loss: … | lr: 0.00000
Done in 2.9 minutes | train loss 1.5129 | held-out loss 2.2712
Saved trained model to …/src/homer_ai/homer_gpt.pt
```

Watch the held-out column while it runs. As long as it keeps falling, the model is still learning. If it turned upwards, as in run 1 above, you'd stop there.

The run writes `src/homer_ai/homer_gpt.pt` (and replaces the one from an earlier run). The ready-made file in `models/` is never touched. From then on the notebooks print `Using your own GPT`, and the MCP server of [chapter 6](../06_making_it_talk/06_00_chapter_overview.ipynb) picks your file first.

Every checkpoint carries notes on how it was made. So we can read the last line of that log from the ready-made file, without training anything.

👉 **Predict first:** which will be lower, the train loss or the held-out loss?

In [ ]:
from homer_ai.lessons import READY_MADE_GPT

notes = torch.load(READY_MADE_GPT, weights_only=True)["train_meta"]  # read only
for key in ("seed", "epochs", "lr", "batch_size", "train_chars", "val_chars"):
    print(f"{key:12}{notes[key]}")
print(f"train loss    {notes['final_train_loss']:.4f}")
print(f"held-out loss {notes['val_loss']:.4f}")
print(f"time          {notes['train_seconds'] / 60:.1f} minutes")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Seed 42, 200 epochs, peak learning rate 0.001, 16 windows per step, 17,730 letters to learn from and 1,971 held out. The train loss is **1.51** and the held-out loss **2.27**, after 2.9 minutes. Those are the numbers in [models/README.md](../../models/README.md) and in [05_05](05_05_validation.ipynb).

The gap between the two is the price of never having seen the text: some of that 1.51 is Book 1 known by heart. The 2.27 is the number to quote.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Will my run print exactly 1.5129 and 2.2712?

On the same machine and PyTorch version, yes: the run is seeded ([05_04](05_04_running_fair_experiments.ipynb)). On another machine the last digits can differ, because processors don't all round the same way. The time differs more. Three to four minutes is normal on a laptop.

### ❓ Can I train the small model too?

Yes: `uv run homer-train` with no option (about 8 to 10 minutes). It writes `src/homer_ai/homer_model.pt`, see [05_04](05_04_running_fair_experiments.ipynb). The two commands write different files, so neither run disturbs the other.

### ❓ What would I change to get a better model?

More text, first of all. `BOOKS` in `train_homer.py` lists the books to train on, and Book 1 is only about 20,000 letters. With more text a bigger model stops memorising and starts learning, and then a lower dropout and more epochs begin to pay. Change one thing at a time, and watch the held-out loss.

## 6. Recap

- `train_gpt` is the loop you know, fed with batches of random windows. `flatten(0, 1)` turns the `(batch, letters, 137)` scores into one row per question for the loss.
- The cosine schedule ticks once per step. Weight decay pulls every weight a little towards 0, and dropout 0.3 keeps this big model from learning Book 1 by heart.
- `uv run homer-train --model gpt` trains your own in about 3 minutes: train loss 1.51, held-out loss 2.27.

**You can now:**

- read `train_gpt` line by line: batches, schedule, weight decay, dropout
- train your own GPT with `uv run homer-train --model gpt`

**Next:** [chapter 6: Making it talk](../06_making_it_talk/06_00_chapter_overview.ipynb): making the saved model write.